In [ ]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

Cloning into 'mario-the-explorer'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 155 (delta 63), reused 123 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 770.16 KiB | 4.01 MiB/s, done.
Resolving deltas: 100% (63/63), done.


In [ ]:
!sh ./mario-the-explorer/setup.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 156.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 318.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 307.5 MB/s eta 0:00:00
Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [ ]:
!pip install -q stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 8.1 MB/s eta 0:00:00


In [ ]:
from typing import Optional
from enum import Enum
from logging import Logger

import torch
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import KVWriter, Logger as PpoLogger

from mario_the_explorer import MultiAttemptSuperMarioWorldEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger, tile_absolute_id

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
class SuperMarioAction(Enum):
    B = 0
    Y = 1
    SELECT = 2
    START = 3
    UP = 4
    DOWN = 5
    LEFT = 6
    RIGHT = 7
    A = 8
    X = 9
    L = 10
    R = 11

class SuperMarioCombo(Enum):
    DO_NOTHING = []
    LEFT = [SuperMarioAction.LEFT]
    LEFT_RUN = [SuperMarioAction.LEFT, SuperMarioAction.Y]
    LEFT_JUMP = [SuperMarioAction.LEFT, SuperMarioAction.B]
    LEFT_RUN_JUMP = [SuperMarioAction.LEFT, SuperMarioAction.Y, SuperMarioAction.B]
    RIGHT = [SuperMarioAction.RIGHT]
    RIGHT_RUN = [SuperMarioAction.RIGHT, SuperMarioAction.Y]
    RIGHT_JUMP = [SuperMarioAction.RIGHT, SuperMarioAction.B]
    RIGHT_RUN_JUMP = [SuperMarioAction.RIGHT, SuperMarioAction.Y, SuperMarioAction.B]
    JUMP = [SuperMarioAction.B]
    DOWN = [SuperMarioAction.DOWN]

class SuperMarioDiscretizer(gym.ActionWrapper):

    def __init__(self, env):
        super().__init__(env)
        self._action_map = []
        for combo in SuperMarioCombo:
            self._action_map.append(self._build_action_from_combo(combo))
        self.action_space = gym.spaces.Discrete(len(self._action_map))

    def _build_action_from_combo(self, combo: SuperMarioCombo) -> np.ndarray:
        action_vector = np.zeros(len(SuperMarioAction), dtype=np.uint8)
        for action in combo.value:
            action_vector[action.value] = 1
        return action_vector

    def action(self, action: int):
        return self._action_map[int(action)]

In [ ]:
class SeeMoreBlocksRewardModel(RewardModel):
    def __init__(self):
        self._blocks_seen = set()

    def reset(self) -> None:
        self._blocks_seen = set()

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        for row in observation:
            for tile in row:
                tile_id = tile_absolute_id(tile)
                if tile_id not in self._blocks_seen:
                    self._blocks_seen.add(tile_id)
                    return 1.0
        return 0.0

In [ ]:
RUN_NAME = "tile_finder"
LEVEL = "DonutPlains1"
LOG_LEVEL = "INFO"
ATTEMPTS = 3

In [ ]:
def prime_policy_to_go_right(model, logger: Logger, iterations=1000):
    logger.info(f"Priming policy to prefer 'RIGHT'...")
    optimizer = model.policy.optimizer
    right_run_combo_index = list(SuperMarioCombo).index(SuperMarioCombo.RIGHT_RUN_JUMP)
    target_action = torch.tensor([right_run_combo_index]).to(model.device)
    model.policy.train()
    for _ in range(iterations):
        obs, _ = model.policy.obs_to_tensor(env.observation_space.sample())
        distribution = model.policy.get_distribution(obs)
        logits = distribution.distribution.logits
        loss = torch.nn.functional.cross_entropy(logits, target_action)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    logger.info("Priming complete.")

In [ ]:
class PpoKvWriter(KVWriter):
    def __init__(self, logger: Logger):
        self._logger = logger

    def write(self, key_values, key_excluded, step=0):
        for key, value in key_values.items():
            self._logger.info(f"Step {step} - {key}: {value}")

    def close(self):
        pass

In [ ]:
logger = get_file_logger(RUN_NAME, LOG_LEVEL)
ppo_logger = PpoLogger(
    folder=None,
    output_formats=[PpoKvWriter(logger)]
)
base_env = MultiAttemptSuperMarioWorldEmulator(level = LEVEL,
                                               render_mode = "rgb_array",
                                               reward_model = SeeMoreBlocksRewardModel(),
                                               attempts = ATTEMPTS,
                                               render_debug = True,
                                               render_grid = True,
                                               logger = logger)
try:
    env = SuperMarioDiscretizer(base_env)
    env = DummyVecEnv([lambda: env])
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003, n_steps=32768)
    model.set_logger(ppo_logger)
    prime_policy_to_go_right(model, logger)
    logger.info("Starting training...")
    model.learn(total_timesteps=60000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
# finally:
#     env.close()
#     del env
#     import gc
#     gc.collect()

2026-04-26 00:10:14 [INFO] Session log for run tile_finder with level [INFO] initialized at: tile_finder_20260426_001014.log


Using cpu device


2026-04-26 00:10:22 [INFO] Priming policy to prefer 'RIGHT'...
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
2026-04-26 00:10:24 [INFO] Priming complete.
2026-04-26 00:10:24 [INFO] Starting training...
2026-04-26 00:12:54 [INFO] Step 32768 - time/iterations: 1
2026-04-26 00:12:54 [INFO] Step 32768 - time/fps: 218
2026-04-26 00:12:54 [INFO] Step 32768 - time/time_elapsed: 150
2026-04-26 00:12:54 [INFO] Step 32768 - time/total_timesteps: 32768
2026-04-26 00:15:40 [INFO] Step 65536 - train/learning_rate: 0.0003
2026-04-26 00:15:40 [INFO] Step 65536 - train/entropy_loss: -0.558935705415206
2026-04-26 00:15:40 [INFO] Step 65536 - train/policy_gradient_loss: 0.005890074960734637
2026-04-26 00:15:40 [INFO] Step 65536 - tra

In [ ]:
from gymnasium.wrappers import RecordVideo


try:
    env = SuperMarioDiscretizer(base_env)
    env = RecordVideo(env, video_folder="./", name_prefix=f"trial-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.close()
#     del env
#     import gc
#     gc.collect()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-04-26 00:16:10 [INFO] Step: 1000
2026-04-26 00:16:10 [INFO] Terminated: True
2026-04-26 00:16:10 [INFO] Truncated: False
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
try:
    env = SuperMarioDiscretizer(base_env)
    env = DummyVecEnv([lambda: env])
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003, n_steps=32768)
    model.set_logger(ppo_logger)
    prime_policy_to_go_right(model, logger)
    logger.info("Starting training...")
    model.learn(total_timesteps=5000000)
    model.save("ppo_mario_tile_finder")
except Exception as e:
    logger.error(e)
    raise e
# finally:
#     env.close()
#     del env
#     import gc
#     gc.collect()

2026-04-26 00:16:12 [INFO] Priming policy to prefer 'RIGHT'...


Using cpu device


2026-04-26 00:16:14 [INFO] Priming complete.
2026-04-26 00:16:14 [INFO] Starting training...
2026-04-26 00:18:23 [INFO] Step 32768 - train/learning_rate: 0.0003
2026-04-26 00:18:23 [INFO] Step 32768 - train/entropy_loss: -0.5684838536952157
2026-04-26 00:18:23 [INFO] Step 32768 - train/policy_gradient_loss: 0.002766998291326672
2026-04-26 00:18:23 [INFO] Step 32768 - train/value_loss: 0.2645802767226996
2026-04-26 00:18:23 [INFO] Step 32768 - train/approx_kl: 0.012968248687684536
2026-04-26 00:18:23 [INFO] Step 32768 - train/clip_fraction: 0.0726654052734375
2026-04-26 00:18:23 [INFO] Step 32768 - train/loss: 0.11649662256240845
2026-04-26 00:18:23 [INFO] Step 32768 - train/explained_variance: 0.4314652681350708
2026-04-26 00:18:23 [INFO] Step 32768 - train/n_updates: 20
2026-04-26 00:18:23 [INFO] Step 32768 - train/clip_range: 0.2
2026-04-26 00:18:23 [INFO] Step 32768 - time/iterations: 1
2026-04-26 00:18:23 [INFO] Step 32768 - time/fps: 254
2026-04-26 00:18:23 [INFO] Step 32768 - tim

In [ ]:
from gymnasium.wrappers import RecordVideo


try:
    env = SuperMarioDiscretizer(base_env)
    env = RecordVideo(env, video_folder="./", name_prefix=RUN_NAME, episode_trigger=lambda x: True)
    obs, info = env.reset()
    done = False
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.close()
#     del env
#     import gc
#     gc.collect()